# ObjectFormer on TGIF — train on held-out sources, benchmark the resamples

The authors released no usable weights and there is no disk space for the TGIF training split,
so we split `testing/` **by source image**: train on one group of sources, evaluate on the other.
No source appears on both sides, so nothing leaks.

Each resample manifest is then filtered to the held-out sources. Measured against your real
manifests at `HELDOUT_FRACTION = 0.35`:

| | resample 1 | resample 2 | resample 3 |
|---|---|---|---|
| eval images | 1,248 | 1,274 | 1,261 |
| authentic / forged | 96 / 1,152 | 98 / 1,176 | 97 / 1,164 |
| sources | 96 | 98 | 97 |

Training pool: 223 sources ≈ 2,899 images, minus a validation slice.

**Say this in the writeup:** *evaluated on the subset of each resample whose source images were
excluded from training.*

Verified before shipping: model builds without CUDA, 34,413,204 parameters, forward and backward
run, state-dict round-trip gives missing=0 / unexpected=0.

## 0 · Config — edit this cell only

In [1]:
from pathlib import Path

BASE         = Path("/Applications/doc/Data readiness and evaluation in AI")
REPO         = BASE / "ObjectFormer"
TGIF_ROOT    = BASE / "tgif"
MANIFEST_DIR = BASE / "resamples"
OUT_DIR      = BASE / "results" / "benchmark"

HELDOUT_FRACTION = 0.35   # of the 343 source images, held out from training
VAL_FRACTION     = 0.10   # of the TRAINING sources, for monitoring during training
SEED             = 2026

SIZE          = 256
BATCH         = 8
EPOCHS        = 3
LR            = 1e-4
WEIGHT_DECAY  = 0.05
LAMBDA_SEG    = 1.0
MAX_TRAIN_IMAGES = None   # set e.g. 2000 to cut it short
NUM_WORKERS   = 0   # was 4: macOS spawn workers can't unpickle TGIF defined in the notebook

DET_THRESHOLD = 0.5
LOC_THRESHOLD = 0.5

OUT_DIR.mkdir(parents=True, exist_ok=True)
for p in (REPO, TGIF_ROOT, MANIFEST_DIR):
    print(f"{str(p):70s} {p.exists()}")

/Applications/doc/Data readiness and evaluation in AI/ObjectFormer     True
/Applications/doc/Data readiness and evaluation in AI/tgif             True
/Applications/doc/Data readiness and evaluation in AI/resamples        True


## 1 · Imports and device

In [2]:
import sys, time, json, random, re, collections
import numpy as np, cv2, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = (torch.device("cuda") if torch.cuda.is_available()
          else torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cpu"))
print("device:", DEVICE, "| torch", torch.__version__)

device: mps | torch 2.14.0


## 2 · Build the model

In [8]:
sys.path.insert(0, str(REPO))
from ObjectFormer.models.objectformer import ObjectFormer

MODEL_CFG = dict(PRETRAINED="", INPLANES=64,
                 NUM_LAYERS=[1,1,3,1], NUM_CHS=[96,192,384,768], NUM_STRIDES=[1,2,2,2],
                 NUM_HEADS=[3,6,12,24], NUM_PARTS=[64,64,64,64], PATCH_SIZES=[8,7,7,7],
                 DROP_PATH=0.1, NUM_ENC_HEADS=[1,3,6,12], NUM_CLASSES=1)

model = ObjectFormer(MODEL_CFG)
n_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {n_params:,}")
assert n_params == 34_413_204, "parameter count changed — config drift, stop and check"
with torch.no_grad():
    pred, masks = model({"img": torch.randn(1,3,SIZE,SIZE)})
print("detection logit", tuple(pred.shape), "| mask heads", len(masks), "each", tuple(masks[0].shape))
print("masks are sigmoid-activated already -> BCE, not BCEWithLogits")

parameters: 34,413,204


detection logit (1,) | mask heads 3 each (1, 1, 256, 256)
masks are sigmoid-activated already -> BCE, not BCEWithLogits


## 3 · Split the source images

Three disjoint groups of `source_id`: **train**, **val** (monitoring only) and **heldout**
(the evaluation set). Saved to JSON so the split is reproducible and auditable.

In [9]:
man = {k: list(__import__("csv").DictReader(open(MANIFEST_DIR / f"resample_{k}.csv")))
       for k in ("1","2","3")}
for k, rows in man.items():
    print(f"resample {k}: {len(rows)} rows, {len({r['source_id'] for r in rows})} sources")

all_sources = sorted({r["source_id"] for rows in man.values() for r in rows}, key=int)
src_file = MANIFEST_DIR / "resample_sources.csv"
if src_file.exists():
    all_sources = sorted({r["source_id"] for r in __import__("csv").DictReader(open(src_file))}, key=int)
print(f"\ntotal source images: {len(all_sources)}")

rng = random.Random(SEED)
shuffled = list(all_sources); rng.shuffle(shuffled)
n_held = int(len(shuffled) * HELDOUT_FRACTION)
heldout_sources = set(shuffled[:n_held])
pool            = shuffled[n_held:]
n_val           = max(1, int(len(pool) * VAL_FRACTION))
val_sources     = set(pool[:n_val])
train_sources   = set(pool[n_val:])

assert not (heldout_sources & train_sources) and not (heldout_sources & val_sources)
print(f"train {len(train_sources)} | val {len(val_sources)} | heldout {len(heldout_sources)} sources")

json.dump({"seed": SEED, "heldout_fraction": HELDOUT_FRACTION,
           "train_sources": sorted(train_sources, key=int),
           "val_sources": sorted(val_sources, key=int),
           "heldout_sources": sorted(heldout_sources, key=int)},
          open(OUT_DIR / "source_split.json", "w"), indent=1)

for k, rows in man.items():
    h = [r for r in rows if r["source_id"] in heldout_sources]
    lab = collections.Counter(r["label"] for r in h)
    print(f"  resample {k} eval subset: {len(h)} images "
          f"({lab['0']} authentic / {lab['1']} forged), {len({r['source_id'] for r in h})} sources")

resample 1: 3575 rows, 275 sources
resample 2: 3575 rows, 275 sources
resample 3: 3575 rows, 275 sources

total source images: 343
train 201 | val 22 | heldout 120 sources
  resample 1 eval subset: 1287 images (99 authentic / 1188 forged), 99 sources
  resample 2 eval subset: 1248 images (96 authentic / 1152 forged), 96 sources
  resample 3 eval subset: 1235 images (95 authentic / 1140 forged), 95 sources


## 4 · Index the training images

Built from the manifests themselves, so the mask pairing is exactly the one the group defined —
no filename guessing. Rows whose `source_id` is held out are excluded entirely.

In [10]:
by_file = {}
for rows in man.values():
    for r in rows:
        by_file.setdefault(r["file"], r)
print(f"{len(by_file)} unique images across the three resamples")

def make_rows(sources):
    out = []
    for f, r in by_file.items():
        if r["source_id"] not in sources: continue
        img = TGIF_ROOT / f
        mk  = (TGIF_ROOT / r["mask_file"]) if r["mask_file"] else None
        if not img.exists() or (mk is not None and not mk.exists()): continue
        out.append(dict(file=str(img), mask=(str(mk) if mk else None),
                        label=int(r["label"]), generator=r["generator"],
                        source_id=r["source_id"], mask_type=r["mask_type"],
                        category=r["category"], width=int(r["width"]), height=int(r["height"]),
                        rel=f))
    return out

train_rows = make_rows(train_sources)
val_rows   = make_rows(val_sources)
held_rows  = make_rows(heldout_sources)
if MAX_TRAIN_IMAGES:
    rng.shuffle(train_rows); train_rows = train_rows[:MAX_TRAIN_IMAGES]

for name, rr in (("train", train_rows), ("val", val_rows), ("heldout", held_rows)):
    lab = collections.Counter(r["label"] for r in rr)
    print(f"{name:8s} {len(rr):5d} images | {lab[0]} authentic / {lab[1]} forged")

n_pos = sum(r["label"] for r in train_rows); n_neg = len(train_rows) - n_pos
POS_WEIGHT = torch.tensor([n_neg / max(n_pos,1)], dtype=torch.float32, device=DEVICE)
print(f"\ndetection pos_weight = {POS_WEIGHT.item():.4f}")
print("Without it the detection head collapses to 'always forged' — TGIF is ~12 forged per real.")
assert len(train_rows) > 0, "no training rows — check TGIF_ROOT"

4407 unique images across the three resamples
train     2600 images | 200 authentic / 2400 forged
val        273 images | 21 authentic / 252 forged
heldout   1534 images | 118 authentic / 1416 forged



detection pos_weight = 0.0833
Without it the detection head collapses to 'always forged' — TGIF is ~12 forged per real.


## 5 · Dataset

In [11]:
MEAN = np.array([0.485,0.456,0.406], np.float32); STD = np.array([0.229,0.224,0.225], np.float32)

def load_pair(r, size, flip=False):
    img = cv2.imread(r["file"], cv2.IMREAD_COLOR)
    if img is None: raise RuntimeError(f"unreadable: {r['file']}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    if r["mask"]:
        mk = cv2.imread(r["mask"], cv2.IMREAD_GRAYSCALE)
        if mk is None: raise RuntimeError(f"unreadable mask: {r['mask']}")
        mk = (mk > 127).astype(np.float32)
    else:
        mk = np.zeros(img.shape[:2], np.float32)
    img = cv2.resize(img, (size,size), interpolation=cv2.INTER_LINEAR)
    mk  = cv2.resize(mk,  (size,size), interpolation=cv2.INTER_NEAREST)
    if flip: img, mk = img[:, ::-1].copy(), mk[:, ::-1].copy()
    img = (img.astype(np.float32)/255.0 - MEAN) / STD
    return torch.from_numpy(img).permute(2,0,1), torch.from_numpy(mk).unsqueeze(0)

class TGIF(Dataset):
    def __init__(self, rows, size, train): self.rows, self.size, self.train = rows, size, train
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        x, m = load_pair(r, self.size, self.train and random.random() < 0.5)
        return x, m, torch.tensor(float(r["label"]))

train_dl = DataLoader(TGIF(train_rows, SIZE, True),  batch_size=BATCH, shuffle=True,
                      num_workers=NUM_WORKERS, drop_last=True)
val_dl   = DataLoader(TGIF(val_rows,   SIZE, False), batch_size=BATCH, shuffle=False,
                      num_workers=NUM_WORKERS)
xb, mb, yb = next(iter(train_dl))
print("batch", tuple(xb.shape), tuple(mb.shape), tuple(yb.shape),
      f"| mask positive fraction {mb.mean().item():.4f}")

batch (8, 3, 256, 256) (8, 1, 256, 256) (8,) | mask positive fraction 0.0739


## 6 · Train

`L = L_cls + lambda_seg * L_seg`, both binary cross-entropy, lambda_seg = 1, as in the paper.
All three mask heads are supervised and averaged.

In [12]:
model = model.to(DEVICE)
opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
steps_total = EPOCHS * max(len(train_dl), 1)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(steps_total,1))
EPS = 1e-6

def compute_loss(pred, masks, y, m):
    l_cls = F.binary_cross_entropy_with_logits(pred.view(-1), y.view(-1), pos_weight=POS_WEIGHT)
    l_seg = sum(F.binary_cross_entropy(mk.clamp(EPS,1-EPS), m) for mk in masks) / len(masks)
    return l_cls + LAMBDA_SEG*l_seg, l_cls.item(), l_seg.item()

@torch.no_grad()
def quick_eval(dl):
    model.eval(); sc, lb, iou = [], [], []
    for xb, mb, yb in dl:
        xb, mb = xb.to(DEVICE), mb.to(DEVICE)
        pred, masks = model({"img": xb})
        sc += torch.sigmoid(pred.view(-1)).float().cpu().tolist(); lb += yb.view(-1).tolist()
        p = (masks[-1] > LOC_THRESHOLD).float()
        inter = (p*mb).sum((1,2,3)); union = ((p+mb) > 0).float().sum((1,2,3))
        for j, l in enumerate(yb.view(-1).tolist()):
            if l == 1: iou.append((inter[j]/union[j].clamp(min=1)).item())
    model.train()
    auc = roc_auc_score(lb, sc) if len(set(lb)) > 1 else float("nan")
    return auc, (float(np.mean(iou)) if iou else float("nan")), sc

history, step, t0 = [], 0, time.time()
model.train()
for ep in range(EPOCHS):
    for xb, mb, yb in train_dl:
        xb, mb, yb = xb.to(DEVICE), mb.to(DEVICE), yb.to(DEVICE)
        loss, lc, ls = compute_loss(*model({"img": xb}), yb, mb)
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step(); sched.step(); step += 1
        if step % 20 == 0:
            el = time.time()-t0
            print(f"ep{ep+1} {step}/{steps_total} loss {loss.item():.4f} "
                  f"(cls {lc:.4f} seg {ls:.4f}) {el/step:.2f}s/step "
                  f"ETA {(el/step*(steps_total-step))/60:.0f}min")
    auc, iou, _ = quick_eval(val_dl)
    history.append(dict(epoch=ep+1, val_auc=auc, val_iou=iou))
    print(f"\n== epoch {ep+1}: val AUC {auc:.4f} | val IoU {iou:.4f} ==\n")
    torch.save(model.state_dict(), OUT_DIR / f"objectformer_tgif_ep{ep+1}.pth")
print(f"training time: {(time.time()-t0)/60:.1f} min")

ep1 20/975 loss 0.2960 (cls 0.1772 seg 0.1188) 1.19s/step ETA 19min


ep1 40/975 loss 0.4542 (cls 0.2847 seg 0.1695) 1.18s/step ETA 18min


ep1 60/975 loss 0.2533 (cls 0.0633 seg 0.1899) 1.17s/step ETA 18min


ep1 80/975 loss 0.1813 (cls 0.0223 seg 0.1590) 1.17s/step ETA 17min


ep1 100/975 loss 0.4262 (cls 0.1448 seg 0.2814) 1.16s/step ETA 17min


...

ep3 920/975 loss 0.3704 (cls 0.2248 seg 0.1456) 1.23s/step ETA 1min


ep3 940/975 loss 0.3403 (cls 0.1401 seg 0.2002) 1.23s/step ETA 1min


ep3 960/975 loss 0.3149 (cls 0.2150 seg 0.0999) 1.23s/step ETA 0min



== epoch 3: val AUC 0.5058 | val IoU 0.0326 ==



training time: 20.2 min


## 7 · Save and check the model isn't degenerate

In [13]:
ckpt = OUT_DIR / "objectformer_tgif.pth"
torch.save(model.state_dict(), ckpt)
auc, iou, sc = quick_eval(val_dl); s = np.array(sc)
print(f"saved {ckpt}  ({len(model.state_dict())} keys, no 'module.' prefix)")
print(f"val AUC {auc:.4f} | val IoU {iou:.4f}")
print(f"score spread: min {s.min():.4f} max {s.max():.4f} mean {s.mean():.4f} std {s.std():.4f}")
DEGENERATE = s.std() < 1e-3
print("\n*** DEGENERATE — constant output, do not report these numbers ***" if DEGENERATE
      else "\nNot degenerate.")
json.dump({"history": history, "n_params": n_params, "size": SIZE, "epochs": EPOCHS, "lr": LR,
           "batch": BATCH, "lambda_seg": LAMBDA_SEG, "pos_weight": POS_WEIGHT.item(),
           "n_train": len(train_rows), "n_val": len(val_rows), "n_heldout": len(held_rows),
           "heldout_fraction": HELDOUT_FRACTION, "seed": SEED, "degenerate": bool(DEGENERATE)},
          open(OUT_DIR / "training_run.json", "w"), indent=1)

saved /Applications/doc/Data readiness and evaluation in AI/results/benchmark/objectformer_tgif.pth  (427 keys, no 'module.' prefix)
val AUC 0.5058 | val IoU 0.0326
score spread: min 0.5192 max 0.5247 mean 0.5216 std 0.0013

Not degenerate.


## 8 · Inference over the held-out images

One forward pass per unique image, then joined onto each resample. Scored at each image's
**native** resolution, as the paper does: the predicted mask is upsampled back to width x height.

In [14]:
@torch.no_grad()
def predict_all(rows):
    model.eval(); out = {}; t0 = time.time()
    for i in range(0, len(rows), BATCH):
        chunk = rows[i:i+BATCH]
        xb = torch.stack([load_pair(r, SIZE)[0] for r in chunk]).to(DEVICE)
        pred, masks = model({"img": xb})
        probs = torch.sigmoid(pred.view(-1)).float().cpu().numpy()
        pm = masks[-1].float().cpu().numpy()
        for j, r in enumerate(chunk):
            big = cv2.resize(pm[j,0], (r["width"], r["height"]), interpolation=cv2.INTER_LINEAR)
            p = (big > LOC_THRESHOLD)
            if r["label"] == 1:
                gt = cv2.imread(r["mask"], cv2.IMREAD_GRAYSCALE) > 127
                if gt.shape != p.shape:
                    gt = cv2.resize(gt.astype(np.uint8), (r["width"], r["height"]),
                                    interpolation=cv2.INTER_NEAREST).astype(bool)
                tp = int((p & gt).sum()); fp = int((p & ~gt).sum()); fn = int((~p & gt).sum())
                out[r["rel"]] = dict(score=float(probs[j]), tp_px=tp, fp_px=fp, fn_px=fn,
                                     pred_area_frac=float(p.mean()), gt_area_frac=float(gt.mean()))
            else:
                out[r["rel"]] = dict(score=float(probs[j]), tp_px=None, fp_px=None, fn_px=None,
                                     pred_area_frac=float(p.mean()), gt_area_frac=0.0)
        if (i//BATCH) % 10 == 0:
            done = i+len(chunk); el = time.time()-t0
            print(f"  {done}/{len(rows)}  {el/max(done,1):.3f}s/img  "
                  f"ETA {(el/max(done,1)*(len(rows)-done))/60:.1f}min")
    model.train(); return out

PRED = predict_all(held_rows)
print(f"\npredicted {len(PRED)} unique held-out images")

  8/1534  0.216s/img  ETA 5.5min


  88/1534  0.077s/img  ETA 1.9min


  168/1534  0.069s/img  ETA 1.6min


  248/1534  0.066s/img  ETA 1.4min


  328/1534  0.066s/img  ETA 1.3min


...

  1288/1534  0.060s/img  ETA 0.2min


  1368/1534  0.060s/img  ETA 0.2min


  1448/1534  0.060s/img  ETA 0.1min


  1528/1534  0.060s/img  ETA 0.0min



predicted 1534 unique held-out images


## 9 · Dataframes — three CSVs per resample

In [15]:
import pandas as pd
META = ["resample_id","image_id","source_id","category","generator","mask_type","variant","label"]

def metrics_for(rows_man, rid):
    loc, det = [], []
    for r in rows_man:
        p = PRED.get(r["file"])
        if p is None: continue
        base = {k: r.get(k) for k in META}; base["resample_id"] = rid
        det.append({**base, "score": p["score"],
                    "pred": int(p["score"] > DET_THRESHOLD), "threshold": DET_THRESHOLD})
        if int(r["label"]) == 1:
            tp, fp, fn = p["tp_px"], p["fp_px"], p["fn_px"]
            inter = tp; union = tp + fp + fn
            loc.append({**base,
                        "iou":  tp/union if union else 0.0,
                        "dice": 2*tp/(2*tp+fp+fn) if (2*tp+fp+fn) else 0.0,
                        "pixel_precision": tp/(tp+fp) if (tp+fp) else 0.0,
                        "pixel_recall":    tp/(tp+fn) if (tp+fn) else 0.0,
                        "tp_px": tp, "fp_px": fp, "fn_px": fn,
                        "pred_area_frac": p["pred_area_frac"],
                        "gt_area_frac":   p["gt_area_frac"],
                        "threshold": LOC_THRESHOLD})
    return pd.DataFrame(loc), pd.DataFrame(det)

def summarise(det_df, rid):
    out = []
    def block(sub, name):
        y = sub["label"].astype(int).values; s = sub["score"].values
        if len(y) == 0: return
        auc = roc_auc_score(y, s) if len(set(y)) > 1 else float("nan")
        row = dict(resample_id=rid, slice=name, n=len(y),
                   n_forged=int(y.sum()), n_authentic=int((1-y).sum()), roc_auc=auc)
        for tag, thr in (("", DET_THRESHOLD), ("_bestf1", None)):
            if thr is None:
                cands = np.unique(s); best = (-1, DET_THRESHOLD)
                for t in cands:
                    pr = (s > t).astype(int)
                    tp = int(((pr==1)&(y==1)).sum()); fp = int(((pr==1)&(y==0)).sum())
                    fn = int(((pr==0)&(y==1)).sum())
                    f1 = 2*tp/(2*tp+fp+fn) if (2*tp+fp+fn) else 0
                    if f1 > best[0]: best = (f1, t)
                thr = best[1]
            pr = (s > thr).astype(int)
            tp = int(((pr==1)&(y==1)).sum()); fp = int(((pr==1)&(y==0)).sum())
            tn = int(((pr==0)&(y==0)).sum()); fn = int(((pr==0)&(y==1)).sum())
            pcs = tp/(tp+fp) if (tp+fp) else 0.0; rec = tp/(tp+fn) if (tp+fn) else 0.0
            row.update({f"threshold{tag}": float(thr),
                        f"accuracy{tag}": (tp+tn)/len(y),
                        f"precision{tag}": pcs, f"recall{tag}": rec,
                        f"f1{tag}": 2*pcs*rec/(pcs+rec) if (pcs+rec) else 0.0,
                        f"tp{tag}": tp, f"fp{tag}": fp, f"tn{tag}": tn, f"fn{tag}": fn})
        out.append(row)
    block(det_df, "all")
    for g in sorted(det_df["generator"].unique()): block(det_df[det_df.generator==g], f"generator={g}")
    for m in sorted(det_df["mask_type"].unique()): block(det_df[det_df.mask_type==m], f"mask_type={m}")
    y = det_df["label"].astype(int).values
    tp = int(y.sum()); fp = int((1-y).sum())
    pcs = tp/(tp+fp); rec = 1.0
    out.append(dict(resample_id=rid, slice="always_forged", n=len(y), n_forged=tp, n_authentic=fp,
                    roc_auc=0.5, threshold=float("nan"), accuracy=tp/len(y), precision=pcs,
                    recall=rec, f1=2*pcs*rec/(pcs+rec), tp=tp, fp=fp, tn=0, fn=0))
    return pd.DataFrame(out)

SUMMARIES = {}
for k in ("1","2","3"):
    rows_h = [r for r in man[k] if r["source_id"] in heldout_sources]
    loc_df, det_df = metrics_for(rows_h, k)
    sm = summarise(det_df, k); SUMMARIES[k] = sm
    loc_df.to_csv(OUT_DIR/f"resample_{k}_localization.csv", index=False)
    det_df.to_csv(OUT_DIR/f"resample_{k}_detection.csv",    index=False)
    sm.to_csv(OUT_DIR/f"resample_{k}_summary.csv",          index=False)
    a  = sm[sm.slice=="all"].iloc[0]; bl = sm[sm.slice=="always_forged"].iloc[0]
    print(f"\n=== resample {k}: {len(det_df)} images, {len(loc_df)} forged ===")
    print(f"  detection  AUC {a.roc_auc:.4f} | acc {a.accuracy:.4f} F1 {a.f1:.4f} @0.5")
    print(f"  baseline   always-forged acc {bl.accuracy:.4f} F1 {bl.f1:.4f}")
    print("  -> " + ("BEATS the baseline on F1" if a.f1 > bl.f1
                     else "DOES NOT beat the always-forged baseline — NOT A RESULT"))
    print(f"  localization mean IoU {loc_df.iou.mean():.4f} | Dice {loc_df.dice.mean():.4f} "
          f"| pixP {loc_df.pixel_precision.mean():.4f} pixR {loc_df.pixel_recall.mean():.4f}")
    print(f"  mask coverage: mean {loc_df.gt_area_frac.mean():.4f} "
          f"median {loc_df.gt_area_frac.median():.4f}")
    for mt in sorted(loc_df.mask_type.unique()):
        sub = loc_df[loc_df.mask_type==mt]
        print(f"     {mt}: IoU {sub.iou.mean():.4f}, coverage {sub.gt_area_frac.mean():.4f}")


=== resample 1: 1287 images, 1188 forged ===
  detection  AUC 0.5051 | acc 0.9231 F1 0.9600 @0.5
  baseline   always-forged acc 0.9231 F1 0.9600
  -> DOES NOT beat the always-forged baseline — NOT A RESULT
  localization mean IoU 0.0337 | Dice 0.0548 | pixP 0.1738 pixR 0.0366
  mask coverage: mean 0.0586 median 0.0300
     bbox: IoU 0.0326, coverage 0.0762
     segm: IoU 0.0348, coverage 0.0410

=== resample 2: 1248 images, 1152 forged ===
  detection  AUC 0.5046 | acc 0.9231 F1 0.9600 @0.5
  baseline   always-forged acc 0.9231 F1 0.9600
  -> DOES NOT beat the always-forged baseline — NOT A RESULT
  localization mean IoU 0.0287 | Dice 0.0464 | pixP 0.1409 pixR 0.0315
  mask coverage: mean 0.0585 median 0.0285
     bbox: IoU 0.0296, coverage 0.0767
     segm: IoU 0.0278, coverage 0.0403



=== resample 3: 1235 images, 1140 forged ===
  detection  AUC 0.5033 | acc 0.9231 F1 0.9600 @0.5
  baseline   always-forged acc 0.9231 F1 0.9600
  -> DOES NOT beat the always-forged baseline — NOT A RESULT
  localization mean IoU 0.0304 | Dice 0.0489 | pixP 0.1626 pixR 0.0336
  mask coverage: mean 0.0567 median 0.0311
     bbox: IoU 0.0300, coverage 0.0749
     segm: IoU 0.0307, coverage 0.0385


## 10 · Cross-resample stability

The three resamples are 80% draws from the same source pool and share most of it, so they are
**not** independent replicates. No standard deviation across the three. Intervals come from
bootstrapping *within* one resample, resampling by `source_id`.

In [16]:
lines = ["# Cross-resample stability", ""]
rows = []
for k in ("1","2","3"):
    sm = SUMMARIES[k]; a = sm[sm.slice=="all"].iloc[0]
    loc = pd.read_csv(OUT_DIR/f"resample_{k}_localization.csv")
    d = dict(resample=k, det_auc=a.roc_auc, det_f1=a.f1, iou=loc.iou.mean(), dice=loc.dice.mean())
    for g in ("SD","PS"):
        s = sm[sm.slice==f"generator={g}"]
        d[f"auc_{g}"] = float(s.iloc[0].roc_auc) if len(s) else float("nan")
        d[f"iou_{g}"] = loc[loc.generator==g].iou.mean()
    for mt in ("bbox","segm"):
        d[f"iou_{mt}"] = loc[loc.mask_type==mt].iou.mean()
    rows.append(d)
stab = pd.DataFrame(rows)
print(stab.to_string(index=False))
lines += ["```", stab.to_string(index=False), "```", ""]

def ordering(col_a, col_b, la, lb):
    signs = {("<" if r[col_a] < r[col_b] else ">") for _, r in stab.iterrows()}
    ok = len(signs) == 1
    lines.append(f"- {la} vs {lb}: ordering {'HOLDS' if ok else 'DOES NOT HOLD'} "
                 f"in all three ({', '.join(f'{r[col_a]:.4f} vs {r[col_b]:.4f}' for _, r in stab.iterrows())})")
ordering("iou_SD","iou_PS","IoU SD","IoU PS")
ordering("iou_bbox","iou_segm","IoU bbox","IoU segm")
ordering("auc_SD","auc_PS","AUC SD","AUC PS")

loc1 = pd.read_csv(OUT_DIR/"resample_1_localization.csv")
srcs = loc1.source_id.unique(); boot = []
rb = np.random.default_rng(SEED)
for _ in range(2000):
    pick = rb.choice(srcs, size=len(srcs), replace=True)
    boot.append(pd.concat([loc1[loc1.source_id==s] for s in pick]).iou.mean())
lo, hi = np.percentile(boot, [2.5, 97.5])
lines += ["", f"- Resample 1 mean IoU {loc1.iou.mean():.4f}, 95% CI [{lo:.4f}, {hi:.4f}] "
              f"(2000 bootstraps resampled by source_id, n={len(srcs)} sources)"]
(OUT_DIR/"stability.md").write_text("\n".join(lines))
print("\n".join(lines[-5:]))
print("\nwrote", OUT_DIR/"stability.md")

resample  det_auc  det_f1      iou     dice  auc_SD   iou_SD  auc_PS   iou_PS  iou_bbox  iou_segm
       1 0.505085    0.96 0.033723 0.054843     NaN 0.031062     NaN 0.036384  0.032644  0.034803
       2 0.504643    0.96 0.028694 0.046403     NaN 0.026328     NaN 0.031061  0.029578  0.027811
       3 0.503319    0.96 0.030357 0.048949     NaN 0.030791     NaN 0.029923  0.029988  0.030726


- IoU SD vs IoU PS: ordering DOES NOT HOLD in all three (0.0311 vs 0.0364, 0.0263 vs 0.0311, 0.0308 vs 0.0299)
- IoU bbox vs IoU segm: ordering DOES NOT HOLD in all three (0.0326 vs 0.0348, 0.0296 vs 0.0278, 0.0300 vs 0.0307)
- AUC SD vs AUC PS: ordering HOLDS in all three (nan vs nan, nan vs nan, nan vs nan)

- Resample 1 mean IoU 0.0337, 95% CI [0.0209, 0.0482] (2000 bootstraps resampled by source_id, n=99 sources)

wrote /Applications/doc/Data readiness and evaluation in AI/results/benchmark/stability.md


## 11 · Plain-English summary

In [17]:
out = ["# ObjectFormer on TGIF — summary", "",
       f"Trained by us on {len(train_rows)} images from {len(train_sources)} source images; "
       f"evaluated on the subset of each resample whose source images were excluded from "
       f"training ({len(heldout_sources)} held-out sources).",
       f"Input {SIZE}px, {EPOCHS} epochs, AdamW lr {LR}, pos_weight {POS_WEIGHT.item():.3f}, "
       f"{n_params:,} parameters.", ""]
if DEGENERATE:
    out += ["**The model is degenerate — it outputs a constant score. Do not report these "
            "numbers.**", ""]
for k in ("1","2","3"):
    sm = SUMMARIES[k]; a = sm[sm.slice=="all"].iloc[0]; bl = sm[sm.slice=="always_forged"].iloc[0]
    loc = pd.read_csv(OUT_DIR/f"resample_{k}_localization.csv")
    beat = a.f1 > bl.f1
    out += [f"**Resample {k}** ({int(a.n)} images, {int(a.n_forged)} forged / "
            f"{int(a.n_authentic)} authentic)",
            f"- detection ROC-AUC {a.roc_auc:.4f}; accuracy {a.accuracy:.4f}, F1 {a.f1:.4f} at 0.5",
            f"- always-forged baseline: accuracy {bl.accuracy:.4f}, F1 {bl.f1:.4f} "
            f"-> {'beats it' if beat else '**does not beat it — not a result**'}",
            f"- localization mean IoU {loc.iou.mean():.4f}, Dice {loc.dice.mean():.4f}, "
            f"pixel precision {loc.pixel_precision.mean():.4f}, recall {loc.pixel_recall.mean():.4f}",
            ""]
out += ["ROC-AUC is the number to lead with: the classes are about 12 forged per authentic "
        "image, so accuracy and F1 are inflated by the imbalance.", "",
        "Caveat to state: the authors released no usable weights, so this is ObjectFormer "
        "trained by us, not the published model. The training and evaluation images share no "
        "source image, so nothing leaks, but the evaluation subset is smaller than the full "
        "resample."]
(OUT_DIR/"summary.md").write_text("\n".join(out))
print("\n".join(out))

# ObjectFormer on TGIF — summary

Trained by us on 2600 images from 201 source images; evaluated on the subset of each resample whose source images were excluded from training (120 held-out sources).
Input 256px, 3 epochs, AdamW lr 0.0001, pos_weight 0.083, 34,413,204 parameters.

**Resample 1** (1287 images, 1188 forged / 99 authentic)
- detection ROC-AUC 0.5051; accuracy 0.9231, F1 0.9600 at 0.5
- always-forged baseline: accuracy 0.9231, F1 0.9600 -> **does not beat it — not a result**
- localization mean IoU 0.0337, Dice 0.0548, pixel precision 0.1738, recall 0.0366

**Resample 2** (1248 images, 1152 forged / 96 authentic)
- detection ROC-AUC 0.5046; accuracy 0.9231, F1 0.9600 at 0.5
- always-forged baseline: accuracy 0.9231, F1 0.9600 -> **does not beat it — not a result**
- localization mean IoU 0.0287, Dice 0.0464, pixel precision 0.1409, recall 0.0315

**Resample 3** (1235 images, 1140 forged / 95 authentic)
- detection ROC-AUC 0.5033; accuracy 0.9231, F1 0.9600 at 0.5
- always-